# 01. 大模型 API 与消息格式

AGDR VLA 入门教程 · 第 3 章 · 前半周

- **学习目标**：能手写 OpenAI 和 Anthropic 两种格式的请求，看懂返回结果，理解「工具调用」在消息层面是怎么一来一回的
- **前置知识**：[00-导读与学习计划](./00-导读与学习计划.ipynb)；会一点 Python 的字典和列表
- **预计学习时长**：1.5 小时

## 先认识几类 AI 工具

| 类型 | 例子 | 你怎么用它 |
| --- | --- | --- |
| 聊天助手 | 网页版 ChatGPT、Claude、DeepSeek | 你提问，它回答；动手的人是你 |
| 代码补全 | IDE 里的灰色补全提示 | 你写代码，它猜下一段 |
| 编程 Agent | Claude Code、Codex | 你给目标，它自己读文件、运行命令、改代码，直到完成 |

三者的背后是**同一种东西**：通过 HTTP 调用一个大模型 API，发送一段 JSON，收到一段 JSON。区别只在于外面包了多少程序逻辑。

本节先把这个最底层的 API 看清楚，下一节就用它亲手写一个 Agent。

## 模型 API 是什么

调用一次模型，本质上就是发送一个 HTTP POST 请求：

```text
你的程序  ──── 请求 JSON：模型名 + 对话历史 + 可用工具 ───▶  模型服务
          ◀─── 响应 JSON：模型的回复 + 停止原因 + token 用量 ───
```

有一条非常重要的性质：**模型是无状态的。** 服务端不会记住你上一次说了什么。你在网页上看到的「多轮对话」，其实是程序每次都把**完整的对话历史**重新发一遍。记住这一点，后面讲上下文压缩时会用到。

目前最常见的接口格式有两种：

- **OpenAI 格式**（Chat Completions）：最早普及，大量服务商和开源推理框架都兼容它
- **Anthropic 格式**（Messages）：Claude 的原生接口，Claude Code 用的就是它，也有不少服务商兼容

两者思路相同，字段细节不同。我们先学 OpenAI 格式，再学 Anthropic 格式，最后做对比。

## 准备工作

下面两个单元格读取 00 节配置的环境变量，并定义几个小工具函数。**必须先运行它们。**

代码只用到 Python 标准库：`urllib.request` 负责发 HTTP 请求，`json` 负责在 Python 字典和 JSON 文本之间转换。

In [1]:
import json
import os
import urllib.error
import urllib.request
from pathlib import Path

OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "")

ANTHROPIC_BASE_URL = os.environ.get("ANTHROPIC_BASE_URL", "https://api.anthropic.com")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "")

# key 和模型名都设置了，才走在线模式
OPENAI_ONLINE = bool(OPENAI_API_KEY and OPENAI_MODEL)
ANTHROPIC_ONLINE = bool(ANTHROPIC_API_KEY and ANTHROPIC_MODEL)

print("OpenAI 格式:   ", "在线" if OPENAI_ONLINE else "离线（读取示例响应）")
print("Anthropic 格式:", "在线" if ANTHROPIC_ONLINE else "离线（读取示例响应）")

OpenAI 格式:    离线（读取示例响应）
Anthropic 格式: 离线（读取示例响应）


In [ ]:
SAMPLES = Path("data/samples")


def show(obj):
    """把字典漂亮地打印成 JSON，中文不转义"""
    print(json.dumps(obj, ensure_ascii=False, indent=2))


def post_json(url, headers, body):
    """发送一个 POST 请求，返回解析后的 JSON"""
    request = urllib.request.Request(
        url,
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json", **headers},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        # 把服务端返回的错误信息原样显示出来，排错时最有用
        detail = error.read().decode("utf-8", "replace")
        raise RuntimeError(f"HTTP {error.code}: {detail}") from None


def call_openai(body, sample):
    """调用 OpenAI 格式接口；离线时读取示例响应 sample"""
    if not OPENAI_ONLINE:
        print(f"[离线模式] 读取 data/samples/{sample}")
        return json.loads((SAMPLES / sample).read_text(encoding="utf-8"))
    url = OPENAI_BASE_URL.rstrip("/") + "/chat/completions"
    headers = {"Authorization": f"Bearer {OPENAI_API_KEY}"}
    return post_json(url, headers, {"model": OPENAI_MODEL, **body})


def call_anthropic(body, sample):
    """调用 Anthropic 格式接口；离线时读取示例响应 sample"""
    if not ANTHROPIC_ONLINE:
        print(f"[离线模式] 读取 data/samples/{sample}")
        return json.loads((SAMPLES / sample).read_text(encoding="utf-8"))
    url = ANTHROPIC_BASE_URL.rstrip("/") + "/v1/messages"
    headers = {"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01"}
    return post_json(url, headers, {"model": ANTHROPIC_MODEL, **body})

注意两个接口的区别已经出现了：

| | OpenAI 格式 | Anthropic 格式 |
| --- | --- | --- |
| 地址 | `{BASE_URL}/chat/completions` | `{BASE_URL}/v1/messages` |
| 鉴权请求头 | `Authorization: Bearer <key>` | `x-api-key: <key>` |
| 额外请求头 | 无 | `anthropic-version: 2023-06-01` |

`anthropic-version` 是接口版本号，不是模型版本，照写即可。

## OpenAI 格式

### 请求体

一个最简单的请求只需要两个字段：`model`（已经在 `call_openai` 里自动加上）和 `messages`。

`messages` 是一个列表，每条消息有 `role` 和 `content`：

| role | 谁说的 | 用途 |
| --- | --- | --- |
| `system` | 开发者 | 设定身份、规则、回答风格，通常放在第一条 |
| `user` | 用户 | 用户的提问 |
| `assistant` | 模型 | 模型之前的回复（多轮对话时由你回传） |
| `tool` | 工具 | 工具的执行结果（后面讲） |

In [ ]:
body = {
    "messages": [
        {"role": "system", "content": "你是机器人实验室的助教，用一句话回答。"},
        {"role": "user", "content": "什么是 VLA 模型？"},
    ],
}
reply = call_openai(body, "openai_hello.json")
show(reply)

### 读懂响应

响应里最重要的是 `choices[0]`：

- `message`：模型的回复，`role` 固定是 `assistant`
- `finish_reason`：模型为什么停下。`stop` 表示正常说完；`length` 表示达到了长度上限，回答被截断；`tool_calls` 表示模型想调用工具
- `usage`：这次消耗的 token 数，计费就按它算。`prompt_tokens` 是输入，`completion_tokens` 是输出

`choices` 是列表，是因为请求时可以让模型一次生成多个候选回答，平时只用第一个。

In [ ]:
message = reply["choices"][0]["message"]
print("回复:", message["content"])
print("停止原因:", reply["choices"][0]["finish_reason"])
print("token 用量:", reply["usage"])

### 多轮对话：模型不记得你

想继续追问，就要把**上一轮的回复也放回 `messages`**，再加上新问题，整个列表重新发送。

In [ ]:
history = body["messages"] + [
    message,  # 上一轮模型的回复，原样放回
    {"role": "user", "content": "举一个具体的例子。"},
]
print("这次发送的消息条数:", len(history))

reply2 = call_openai({"messages": history}, "openai_followup.json")
print("回复:", reply2["choices"][0]["message"]["content"])
print("输入 token:", reply2["usage"]["prompt_tokens"])

对比两次的 `prompt_tokens`：第二次明显变多了，因为整段历史又发了一遍。**对话越长，每一次调用就越慢、越贵**，直到超出模型的上下文窗口。这正是第 03 节要解决的问题。

> 在线模式下，试试删掉 `message` 那一行再发送：模型不知道你说的「例子」指什么。

### 工具调用（Function Calling）

模型本身只能输出文字，不能查数据库、读文件、控制机器人。**工具调用**让模型可以「请求」你的程序替它做事：

```text
① 你：问题 + 工具说明书              ──▶ 模型
② 模型：「请帮我调用 X，参数是 Y」   ◀──
③ 你的程序真的去执行 X(Y)，得到结果
④ 你：之前的全部消息 + 执行结果      ──▶ 模型
⑤ 模型：根据结果给出最终回答         ◀──
```

关键在第 ③ 步：**工具是你的程序执行的，不是模型执行的。** 模型只负责决定调哪个工具、传什么参数。

我们用一个假的工具演示：查询实验室机器人的电量。

In [ ]:
def query_robot_battery(robot_id):
    """假的工具：返回机器人电量。真实场景里它可能去读机器人的状态接口。"""
    fake_data = {"so100-01": 87, "so100-02": 23}
    if robot_id not in fake_data:
        return {"error": f"找不到机器人 {robot_id}"}
    return {"robot_id": robot_id, "battery_percent": fake_data[robot_id]}


print(query_robot_battery("so100-02"))

要让模型知道有这个工具，需要在请求里附上 `tools`，也就是工具的「说明书」。

`parameters` 用 **JSON Schema** 描述参数：参数叫什么、是什么类型、哪些必填。`description` 写得越清楚，模型越知道什么时候该用它。

In [ ]:
openai_tools = [
    {
        "type": "function",
        "function": {
            "name": "query_robot_battery",
            "description": "查询实验室某台机器人的当前电量（百分比）",
            "parameters": {
                "type": "object",
                "properties": {
                    "robot_id": {"type": "string", "description": "机器人编号，例如 so100-01"},
                },
                "required": ["robot_id"],
            },
        },
    }
]

messages = [{"role": "user", "content": "so100-02 还有多少电？需要充电吗？"}]
reply = call_openai({"messages": messages, "tools": openai_tools}, "openai_tool_call.json")
show(reply["choices"][0])

这次 `finish_reason` 是 `tool_calls`，`content` 是空的，`tool_calls` 里写着模型想调用的工具：

- `id`：这次调用的编号，回传结果时要对上
- `function.name`：工具名
- `function.arguments`：参数。⚠️ 注意它是一个 **JSON 字符串**，不是字典，要先用 `json.loads` 解析

下面完成第 ③、④ 步：执行工具，再把结果发回去。

In [ ]:
assistant_message = reply["choices"][0]["message"]
messages.append(assistant_message)  # 先把模型的「调用请求」放回历史

for call in assistant_message["tool_calls"]:
    args = json.loads(call["function"]["arguments"])  # 字符串 → 字典
    result = query_robot_battery(**args)
    print(f"执行 {call['function']['name']}({args}) → {result}")
    messages.append({
        "role": "tool",
        "tool_call_id": call["id"],  # 对应哪一次调用
        "content": json.dumps(result, ensure_ascii=False),
    })

print("\n现在的消息历史:")
show(messages)

In [ ]:
final = call_openai({"messages": messages, "tools": openai_tools}, "openai_tool_final.json")
print("停止原因:", final["choices"][0]["finish_reason"])
print("最终回答:", final["choices"][0]["message"]["content"])

这就是一次完整的工具调用。仔细看最后的消息历史，顺序是固定的：

```text
user       提问
assistant  tool_calls（模型请求调用）
tool       执行结果（tool_call_id 对应上面的 id）
assistant  最终回答
```

如果模型一次请求了多个工具，`tool_calls` 里就有多项，每一项都要回传一条 `role: tool` 的消息。

> 离线模式下最终回答是录制好的；在线模式下，试着把假数据里的 `23` 改成 `95` 再运行，看模型的建议会不会变。

## Anthropic 格式

现在用 Anthropic 格式把同样的两件事再做一遍。先看请求体的三个不同点：

1. **`system` 是顶层字段**，不放在 `messages` 里
2. **`max_tokens` 必填**，是本次最多输出的 token 数
3. `messages` 里只有 `user` 和 `assistant` 两种角色，并且要**一问一答交替出现**

In [ ]:
body = {
    "system": "你是机器人实验室的助教，用一句话回答。",
    "max_tokens": 512,
    "messages": [
        {"role": "user", "content": "什么是 VLA 模型？"},
    ],
}
reply = call_anthropic(body, "anthropic_hello.json")
show(reply)

### 读懂响应：content 是一个列表

Anthropic 格式最大的特点是：**`content` 是由「内容块」组成的列表**，每个块有自己的 `type`。纯文字回复就是一个 `text` 块；后面会看到，工具调用也是一种块。

停止原因叫 `stop_reason`：`end_turn` 表示正常说完，`max_tokens` 表示被截断，`tool_use` 表示想调用工具。

In [ ]:
for block in reply["content"]:
    if block["type"] == "text":
        print("回复:", block["text"])
print("停止原因:", reply["stop_reason"])
print("token 用量:", reply["usage"])

### 工具调用

工具定义更简洁：没有外层的 `type` / `function`，参数的 JSON Schema 放在 `input_schema` 里。

In [ ]:
anthropic_tools = [
    {
        "name": "query_robot_battery",
        "description": "查询实验室某台机器人的当前电量（百分比）",
        "input_schema": {
            "type": "object",
            "properties": {
                "robot_id": {"type": "string", "description": "机器人编号，例如 so100-01"},
            },
            "required": ["robot_id"],
        },
    }
]

messages = [{"role": "user", "content": "so100-02 还有多少电？需要充电吗？"}]
reply = call_anthropic(
    {"max_tokens": 512, "messages": messages, "tools": anthropic_tools},
    "anthropic_tool_use.json",
)
print("停止原因:", reply["stop_reason"])
show(reply["content"])

模型的回复里可以**同时有文字和工具调用**：先一个 `text` 块说「我先查一下」，再一个 `tool_use` 块。

- `tool_use.input` 已经是**字典**，不需要 `json.loads`，这一点和 OpenAI 格式不同
- 执行结果要放进一个 `tool_result` 块，用 `tool_use_id` 对应调用编号
- 这个 `tool_result` 块放在一条 **`role: user`** 的消息里。在 Anthropic 格式中，工具结果被看作「用户这一侧」提供的信息，这样一问一答的交替规则就不会被打破

In [ ]:
messages.append({"role": "assistant", "content": reply["content"]})  # 模型的回复原样放回

results = []
for block in reply["content"]:
    if block["type"] == "tool_use":
        result = query_robot_battery(**block["input"])  # 已经是字典
        print(f"执行 {block['name']}({block['input']}) → {result}")
        results.append({
            "type": "tool_result",
            "tool_use_id": block["id"],
            "content": json.dumps(result, ensure_ascii=False),
        })

messages.append({"role": "user", "content": results})  # 所有结果放在同一条 user 消息里

final = call_anthropic(
    {"max_tokens": 512, "messages": messages, "tools": anthropic_tools},
    "anthropic_tool_final.json",
)
print("停止原因:", final["stop_reason"])
print("最终回答:", final["content"][0]["text"])

## 两种格式对比

| | OpenAI（Chat Completions） | Anthropic（Messages） |
| --- | --- | --- |
| 系统提示词 | `messages` 里 `role: system` 的一条 | 顶层字段 `system` |
| `max_tokens` | 可选 | **必填** |
| 消息角色 | `system` / `user` / `assistant` / `tool` | `user` / `assistant`，交替出现 |
| 回复内容 | `content` 是字符串 | `content` 是内容块列表 |
| 工具定义 | `{type: function, function: {name, description, parameters}}` | `{name, description, input_schema}` |
| 模型请求调用工具 | `message.tool_calls`，参数是 **JSON 字符串** | `tool_use` 块，参数 `input` 是**字典** |
| 回传工具结果 | 每个结果一条 `role: tool` 消息，带 `tool_call_id` | 一条 `role: user` 消息，内含 `tool_result` 块，带 `tool_use_id` |
| 停止原因 | `finish_reason`：`stop` / `length` / `tool_calls` | `stop_reason`：`end_turn` / `max_tokens` / `tool_use` |
| token 用量 | `prompt_tokens` / `completion_tokens` | `input_tokens` / `output_tokens` |

记住一句话：**结构不同，流程完全一样**。都是「发历史 → 模型要调工具 → 执行 → 回传结果 → 再发历史」。下一节的 Agent Loop，就是把这个流程放进一个循环里。

## Token 与上下文窗口

**token** 是模型处理文字的最小单位，可以粗略理解为「一小块文字」。英文里一个常见单词大约是一个 token；中文的换算比例因模型的分词器而异。想知道准确数字，就看响应里的 `usage`。

token 决定了两件事：

- **花费**：按输入和输出的 token 数计费，输出通常比输入贵
- **上限**：每个模型有一个**上下文窗口**，即一次请求里输入加输出的 token 总数上限。超过就会报错，或者必须丢掉一部分历史

再回想「模型无状态」：Agent 每调用一次工具，消息历史就长一截，下一次请求就要多发这些 token。一个运行了几十步的 Agent，消息历史很容易涨到几万 token。这就是第 03 节「上下文压缩」要解决的问题。

## 流式输出

网页上的回答是一个字一个字「蹦」出来的，这叫**流式输出**。请求时加上 `"stream": true`，服务端就不再一次性返回完整 JSON，而是通过 SSE（Server-Sent Events）持续推送一小段一小段的增量：

- OpenAI 格式：每段是一个 chunk，新增的文字在 `choices[0].delta` 里
- Anthropic 格式：按事件推送，例如 `message_start`、`content_block_delta`、`message_stop`

流式输出只改变「怎么收」，不改变消息格式本身。本章的代码为了好读，一律使用非流式请求。

## 用官方 SDK 怎么写

实际项目里一般不手写 HTTP 请求，而是用官方 SDK。SDK 帮你处理地址、请求头、重试和流式解析，但**请求体的结构和上面完全一样**。下面的代码只是展示，本章不要求安装（第 4 章讲 Python 环境和 `pip`）：

```python
from openai import OpenAI          # pip install openai
client = OpenAI()                  # 自动读取 OPENAI_API_KEY、OPENAI_BASE_URL
reply = client.chat.completions.create(
    model="模型名",
    messages=[{"role": "user", "content": "什么是 VLA 模型？"}],
)
print(reply.choices[0].message.content)
```

```python
import anthropic                   # pip install anthropic
client = anthropic.Anthropic()     # 自动读取 ANTHROPIC_API_KEY、ANTHROPIC_BASE_URL
reply = client.messages.create(
    model="模型名",
    max_tokens=512,
    messages=[{"role": "user", "content": "什么是 VLA 模型？"}],
)
print(reply.content[0].text)
```

> 补充：OpenAI 后来还推出了 **Responses API**（地址是 `/responses`），字段名有所不同。因为兼容 Chat Completions 的服务最多，本教程以 Chat Completions 为准。遇到新接口时，按本节的方法读一遍官方文档里的请求和响应示例即可。

## 动手试试

不计分。前三题在离线模式下也能完成。

1. 下面是一段 OpenAI 格式的对话历史，把它改写成 Anthropic 格式的请求体（写成 Python 字典）：

   ```python
   [
       {"role": "system", "content": "你是助教。"},
       {"role": "user", "content": "ROS2 是什么？"},
       {"role": "assistant", "content": "ROS2 是机器人软件的开发框架。"},
       {"role": "user", "content": "和 ROS1 有什么区别？"},
   ]
   ```

2. 为工具 `move_joint` 分别写出两种格式的工具定义。它有两个必填参数：`joint_id`（整数，关节编号）和 `angle`（数字，目标角度，单位是度）。

3. 模型在一次回复里请求调用**两个**工具。回传结果时，两种格式分别要往 `messages` 里追加几条消息？每条的 `role` 是什么？

4. （在线）把 Anthropic 示例的 `max_tokens` 改成 `5` 再运行，观察 `stop_reason` 和回复内容。

<details>
<summary>参考答案</summary>

**第 1 题**

```python
{
    "system": "你是助教。",
    "max_tokens": 512,
    "messages": [
        {"role": "user", "content": "ROS2 是什么？"},
        {"role": "assistant", "content": "ROS2 是机器人软件的开发框架。"},
        {"role": "user", "content": "和 ROS1 有什么区别？"},
    ],
}
```

`system` 移到顶层，并补上必填的 `max_tokens`。`content` 是纯字符串时可以直接写字符串，接口会把它当作一个 `text` 块。

**第 2 题**

```python
schema = {
    "type": "object",
    "properties": {
        "joint_id": {"type": "integer", "description": "关节编号"},
        "angle": {"type": "number", "description": "目标角度，单位：度"},
    },
    "required": ["joint_id", "angle"],
}
openai_tool = {
    "type": "function",
    "function": {"name": "move_joint", "description": "把指定关节转到目标角度", "parameters": schema},
}
anthropic_tool = {"name": "move_joint", "description": "把指定关节转到目标角度", "input_schema": schema}
```

两者的 JSON Schema 完全相同，只是外面的包装不同。

**第 3 题**

- OpenAI 格式：共 3 条。1 条 `assistant`（含两项 `tool_calls`），加 2 条 `tool`（每个结果一条，各自带 `tool_call_id`）
- Anthropic 格式：共 2 条。1 条 `assistant`（含两个 `tool_use` 块），加 1 条 `user`（含两个 `tool_result` 块）

**第 4 题**

`stop_reason` 变成 `max_tokens`，回复只有开头几个字就被截断了。Agent 遇到这种情况时，不能把半截回答当作最终结果。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 说出聊天助手、代码补全、编程 Agent 的区别与共同点 | 先认识几类 AI 工具 |
| 解释「模型无状态」，以及多轮对话为什么要回传历史 | 多轮对话 |
| 手写两种格式的请求体，并读出回复、停止原因和 token 用量 | OpenAI 格式、Anthropic 格式 |
| 说清工具调用的五个步骤，以及「工具由谁执行」 | 工具调用 |
| 指出两种格式在系统提示词、工具定义、结果回传上的差异 | 两种格式对比 |
| 解释 token、上下文窗口与花费的关系 | Token 与上下文窗口 |

下一节：[02-手写AgentLoop](./02-手写AgentLoop.ipynb)